# BNEPA Royalty Report Analysis

Source: `BNEPA_Royalty_Report_MAY2026_LV.xlsx` — monthly China royalty statements.

**Task 1:** Extract the distinct list of `Product Name` + `Item Number` across all monthly sheets.

In [1]:
import re

import pandas as pd
from pathlib import Path

from normalize import normalize_name

WORKBOOK = Path("BNEPA_Royalty_Report_MAY2026_LV.xlsx")
SKIP_SHEETS = {"New template", "銷售庫存統計總表"}
HEADER_SCAN_DEPTH = 12  # rows scanned for the 'Product Name' header

In [2]:
def find_header_row(path: Path, sheet: str, depth: int = HEADER_SCAN_DEPTH) -> int:
    """Return the 0-indexed row that contains the 'Product Name' header."""
    probe = pd.read_excel(path, sheet_name=sheet, header=None, nrows=depth, engine="openpyxl")
    for i in range(len(probe)):
        cells = {str(v).strip() for v in probe.iloc[i].tolist() if pd.notna(v)}
        if "Product Name" in cells:
            return i
    raise ValueError(f"No 'Product Name' header found in first {depth} rows of {sheet!r}")

xl = pd.ExcelFile(WORKBOOK, engine="openpyxl")
monthly = [s for s in xl.sheet_names if s not in SKIP_SHEETS]
frames = {}
for s in monthly:
    hdr = find_header_row(WORKBOOK, s)
    frames[s] = pd.read_excel(xl, sheet_name=s, header=hdr, engine="openpyxl")
    print(f"  {s}: header at row {hdr + 1}, {len(frames[s])} data rows")

print(f"\nLoaded {len(frames)} monthly sheets")

  8月9月: header at row 8, 120 data rows


  10月: header at row 8, 123 data rows


  11月: header at row 8, 234 data rows


  12月: header at row 8, 203 data rows


  1月: header at row 8, 177 data rows


...

  12月2025: header at row 7, 463 data rows


  1月2026: header at row 7, 446 data rows


  2月2026 : header at row 7, 508 data rows


  3月2026: header at row 7, 476 data rows


  4月2026: header at row 7, 450 data rows

Loaded 20 monthly sheets


In [3]:
parts = []
for name, df in frames.items():
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn = cols.get("Product Name")
    inum = cols.get("Item Number")
    if not pn or not inum:
        print(f"  [skip] {name}: missing columns ({list(df.columns)[:5]}…)")
        continue
    sub = df[[pn, inum]].rename(columns={pn: "Product Name", inum: "Item Number"})
    sub["sheet"] = name
    parts.append(sub)

combined = pd.concat(parts, ignore_index=True)

combined = combined.dropna(subset=["Product Name", "Item Number"], how="any")
combined["Product Name"] = combined["Product Name"].astype(str).str.strip()
combined["Item Number"] = combined["Item Number"].astype(str).str.strip()
combined = combined[(combined["Product Name"] != "") & (combined["Item Number"] != "")]
combined["Normalized Name"] = combined["Product Name"].map(normalize_name)
combined = combined[combined["Normalized Name"] != ""]

# Workbook stores monthly sheets in chronological order, so position in
# `monthly` doubles as a recency rank. Keep one row per Normalized Name —
# the variant from the most recent sheet that carried it.
sheet_rank = {s: i for i, s in enumerate(monthly)}
combined["_rank"] = combined["sheet"].map(sheet_rank)

distinct = (
    combined.sort_values("_rank")
    .drop_duplicates(subset=["Normalized Name"], keep="last")[
        ["Product Name", "Item Number", "Normalized Name"]
    ]
    .sort_values(["Product Name", "Item Number"])
    .reset_index(drop=True)
)
combined = combined.drop(columns="_rank")

assert distinct["Normalized Name"].is_unique, "Normalized Name column is not unique"
print(f"{len(distinct)} distinct products (by Normalized Name)")
distinct

176 distinct products (by Normalized Name)


,Product Name,Item Number,Normalized Name
0,.hack//G.U. Last Recode,E05367,hack_g_u_last_recode
1,ACE COMBAT 7: SKIES UNKNOWN,E03174,ace_combat_7_skies_unknown
2,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05445,ace_combat_7_skies_unknown_top_gun_maverick
3,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05444,ace_combat_7_skies_unknown_top_gun_maverick_ul...
4,ARMORED CORE VI FIRES OF RUBICON Deluxe Edition,E05838,armored_core_vi_fires_of_rubicon_deluxe
...,...,...,...
171,That Time I Got Reincarnated as a Slime ISEKAI...,E06534,that_time_i_got_reincarnated_as_a_slime_isekai...
172,Towa and the Guardians of the Sacred Tree,E06936,towa_and_the_guardians_of_the_sacred_tree
173,Towa and the Guardians of the Sacred Tree Delu...,L00117,towa_and_the_guardians_of_the_sacred_tree_deluxe
174,We Love Katamari REROLL+ Royal Reverie,E05071,we_love_katamari_reroll_royal_reverie


In [4]:
out = Path("distinct_products.csv")
distinct.to_csv(out, index=False)
print(f"Wrote {out.resolve()}")

Wrote /Users/christopherdrews/dev/bandai-data-analysis/distinct_products.csv


## Task 2 — SRP history per product

For each (Product Name, Item Number), list the SRP (CNY) values it had and the month range each one applied to. Consecutive months at the same SRP collapse into a single range; a gap in sheets (a month where the product didn't sell) breaks the run.

In [5]:
import re
import datetime as _dt

DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}")

def find_sales_period(path: Path, sheet: str, depth: int = 12) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Return (start_date, end_date) for the sheet by scanning rows below 'Sales Period :'."""
    probe = pd.read_excel(path, sheet_name=sheet, header=None, nrows=depth, engine="openpyxl")
    dates: list[pd.Timestamp] = []
    started = False
    for i in range(len(probe)):
        for v in probe.iloc[i].tolist():
            if pd.isna(v):
                continue
            if isinstance(v, str) and "Sales Period" in v:
                started = True
                continue
            if not started:
                continue
            if isinstance(v, (pd.Timestamp, _dt.datetime, _dt.date)):
                dates.append(pd.Timestamp(v))
            elif isinstance(v, str) and DATE_RE.match(v):
                dates.append(pd.Timestamp(v))
            if len(dates) == 2:
                return dates[0], dates[1]
    raise ValueError(f"Could not find sales-period dates in {sheet!r}")

sheet_periods = {s: find_sales_period(WORKBOOK, s) for s in monthly}
sheet_order = sorted(sheet_periods, key=lambda s: sheet_periods[s][0])
for s in sheet_order:
    a, b = sheet_periods[s]
    print(f"  {s}: {a.date()} → {b.date()}")

  8月9月: 2024-08-01 → 2024-09-30
  10月: 2024-10-01 → 2024-10-31
  11月: 2024-11-01 → 2024-11-30
  12月: 2024-12-01 → 2024-12-31
  1月: 2025-01-01 → 2025-01-31
  2月: 2025-02-01 → 2025-02-28
  3月: 2025-03-01 → 2025-03-31
  4月: 2025-04-01 → 2025-04-30
  5月: 2025-05-01 → 2025-05-31
  6月: 2025-06-01 → 2025-06-30
  7月: 2025-07-01 → 2025-07-31
  8月: 2025-08-01 → 2025-08-30
  9月: 2025-09-01 → 2025-09-30
  10月2025: 2025-10-01 → 2025-10-31
  11月2025: 2025-11-01 → 2025-11-30
  12月2025: 2025-12-01 → 2025-12-31
  1月2026: 2026-01-01 → 2026-01-31
  2月2026 : 2026-02-01 → 2026-02-28
  3月2026: 2026-03-01 → 2026-03-31
  4月2026: 2026-04-01 → 2026-04-30


In [6]:
srp_parts = []
for sheet in sheet_order:
    df = frames[sheet]
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn, inum, srp = cols.get("Product Name"), cols.get("Item Number"), cols.get("SRP (CNY)")
    if not (pn and inum and srp):
        print(f"  [skip] {sheet}: missing one of Product Name / Item Number / SRP (CNY)")
        continue
    sub = df[[pn, inum, srp]].rename(
        columns={pn: "Product Name", inum: "Item Number", srp: "SRP (CNY)"}
    )
    sub = sub.dropna(subset=["Product Name", "Item Number", "SRP (CNY)"], how="any")
    sub["Product Name"] = sub["Product Name"].astype(str).str.strip()
    sub["Item Number"] = sub["Item Number"].astype(str).str.strip()
    sub = sub[(sub["Product Name"] != "") & (sub["Item Number"] != "")]
    sub["Normalized Name"] = sub["Product Name"].map(normalize_name)
    sub = sub[sub["Normalized Name"] != ""]
    sub["SRP (CNY)"] = pd.to_numeric(sub["SRP (CNY)"], errors="coerce")
    sub = sub.dropna(subset=["SRP (CNY)"])
    sub["sheet"] = sheet
    sub["period_start"] = sheet_periods[sheet][0]
    sub["period_end"] = sheet_periods[sheet][1]
    srp_parts.append(sub)

srp_long = pd.concat(srp_parts, ignore_index=True)

# Within a single sheet a normalized product can appear on multiple rows
# (different Customer values or trivially different name variants). Keep the
# modal SRP and the display Product Name from the alphabetically-first variant.
srp_long = (
    srp_long.groupby(
        ["Normalized Name", "sheet", "period_start", "period_end"], as_index=False
    ).agg(
        **{
            "Product Name": ("Product Name", "first"),
            "SRP (CNY)": ("SRP (CNY)", lambda s: s.mode().iloc[0]),
        }
    )
)
print(f"{len(srp_long)} (normalized product, sheet) SRP observations")
srp_long.head()

2777 (normalized product, sheet) SRP observations


,Normalized Name,sheet,period_start,period_end,Product Name,SRP (CNY)
0,accel_world_vs_sword_art_online_deluxe,10月,2024-10-01,2024-10-31,Accel World VS. Sword Art Online Deluxe Edition,278.0
1,accel_world_vs_sword_art_online_deluxe,10月2025,2025-10-01,2025-10-31,Accel World VS. Sword Art Online Deluxe Edition,278.0
2,accel_world_vs_sword_art_online_deluxe,11月,2024-11-01,2024-11-30,Accel World VS. Sword Art Online Deluxe Edition,278.0
3,accel_world_vs_sword_art_online_deluxe,11月2025,2025-11-01,2025-11-30,Accel World VS. Sword Art Online Deluxe Edition,278.0
4,accel_world_vs_sword_art_online_deluxe,12月,2024-12-01,2024-12-31,Accel World VS. Sword Art Online Deluxe Edition,278.0


In [7]:
# Sort each product's observations chronologically, then collapse consecutive
# same-SRP rows into ranges. A "gap" (sheet missing between two observations)
# breaks the run.
sheet_idx = {s: i for i, s in enumerate(sheet_order)}
srp_long["_idx"] = srp_long["sheet"].map(sheet_idx)
srp_long = srp_long.sort_values(["Normalized Name", "_idx"]).reset_index(drop=True)

runs = []
for norm, grp in srp_long.groupby("Normalized Name", sort=False):
    grp = grp.sort_values("_idx")
    run_start = run_end = None
    run_srp = None
    prev_idx = None
    for _, row in grp.iterrows():
        if run_srp is None:
            run_srp = row["SRP (CNY)"]
            run_start = row["period_start"]
            run_end = row["period_end"]
            prev_idx = row["_idx"]
            continue
        if row["SRP (CNY)"] == run_srp and row["_idx"] == prev_idx + 1:
            run_end = row["period_end"]
        else:
            runs.append((norm, run_srp, run_start, run_end))
            run_srp = row["SRP (CNY)"]
            run_start = row["period_start"]
            run_end = row["period_end"]
        prev_idx = row["_idx"]
    if run_srp is not None:
        runs.append((norm, run_srp, run_start, run_end))

srp_history = pd.DataFrame(
    runs, columns=["Normalized Name", "SRP (CNY)", "start_date", "end_date"]
)

# Attach the display Product Name from the most recent sheet that carried
# the normalized product, using the same `_rank` map built in Task 1.
display_names = (
    combined.assign(_rank=combined["sheet"].map(sheet_rank))
    .sort_values("_rank")
    .drop_duplicates(subset=["Normalized Name"], keep="last")
    .set_index("Normalized Name")["Product Name"]
)
srp_history.insert(0, "Product Name", srp_history["Normalized Name"].map(display_names))
srp_history["start_month"] = srp_history["start_date"].dt.strftime("%Y-%m")
srp_history["end_month"] = srp_history["end_date"].dt.strftime("%Y-%m")
srp_history = srp_history[
    ["Product Name", "Normalized Name", "SRP (CNY)", "start_month", "end_month",
     "start_date", "end_date"]
].sort_values(["Product Name", "start_date"]).reset_index(drop=True)

print(f"{len(srp_history)} SRP runs across {srp_history['Normalized Name'].nunique()} products")
srp_history.head(20)

319 SRP runs across 176 products


,Product Name,Normalized Name,SRP (CNY),start_month,end_month,start_date,end_date
0,.hack//G.U. Last Recode,hack_g_u_last_recode,278.0,2024-08,2025-04,2024-08-01,2025-04-30
1,.hack//G.U. Last Recode,hack_g_u_last_recode,278.0,2025-06,2026-04,2025-06-01,2026-04-30
2,ACE COMBAT 7: SKIES UNKNOWN,ace_combat_7_skies_unknown,268.0,2024-08,2026-04,2024-08-01,2026-04-30
3,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ace_combat_7_skies_unknown_top_gun_maverick,368.0,2024-08,2024-10,2024-08-01,2024-10-31
4,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ace_combat_7_skies_unknown_top_gun_maverick,368.0,2025-01,2026-04,2025-01-01,2026-04-30
5,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ace_combat_7_skies_unknown_top_gun_maverick_ul...,498.0,2024-08,2024-10,2024-08-01,2024-10-31
6,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ace_combat_7_skies_unknown_top_gun_maverick_ul...,498.0,2025-01,2026-04,2025-01-01,2026-04-30
7,ARMORED CORE VI FIRES OF RUBICON Deluxe Edition,armored_core_vi_fires_of_rubicon_deluxe,348.0,2024-08,2026-04,2024-08-01,2026-04-30
8,ARMORED CORE VI FIRES OF RUBICON Standard Edition,armored_core_vi_fires_of_rubicon,298.0,2024-08,2026-04,2024-08-01,2026-04-30
9,Accel World VS. Sword Art Online Deluxe Edition,accel_world_vs_sword_art_online_deluxe,278.0,2024-08,2026-04,2024-08-01,2026-04-30


In [8]:
out = Path("product_srp_history.csv")
srp_history.drop(columns=["start_date", "end_date"]).to_csv(out, index=False)
print(f"Wrote {out.resolve()}")

Wrote /Users/christopherdrews/dev/bandai-data-analysis/product_srp_history.csv


## Task 3 — Promotion history per (product, customer)

For each (Product Name, Item Number, Customer, Promo Discount), list the month range each promo was active. Customer is from the newer-layout sheets; the older sheets (8月9月 → 6月) have no Customer column and are labelled `All`. Rows where `Promo Discount = 0` (no promo) and aggregate rows (`SUBTOTAL`, `TOTAL`) are excluded.

In [9]:
PROMO_COL_VARIANTS = ("Promo Discount\n(OFF)", "Promo Discount (OFF)")
AGGREGATE_CUSTOMERS = {"SUBTOTAL", "TOTAL"}

promo_parts = []
for sheet in sheet_order:
    df = frames[sheet]
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn = cols.get("Product Name")
    inum = cols.get("Item Number")
    promo = next((cols[k] for k in PROMO_COL_VARIANTS if k in cols), None)
    cust = cols.get("Customer")
    if not (pn and inum and promo):
        print(f"  [skip] {sheet}: missing Product Name / Item Number / Promo Discount")
        continue
    keep = [pn, inum, promo] + ([cust] if cust else [])
    sub = df[keep].copy()
    rename = {pn: "Product Name", inum: "Item Number", promo: "Promo Discount"}
    if cust:
        rename[cust] = "Customer"
    sub = sub.rename(columns=rename)
    if "Customer" not in sub.columns:
        sub["Customer"] = "All"

    sub = sub.dropna(subset=["Product Name", "Item Number", "Promo Discount"], how="any")
    sub["Product Name"] = sub["Product Name"].astype(str).str.strip()
    sub["Item Number"] = sub["Item Number"].astype(str).str.strip()
    sub["Customer"] = sub["Customer"].fillna("All").astype(str).str.strip()
    sub = sub[(sub["Product Name"] != "") & (sub["Item Number"] != "")]
    sub["Normalized Name"] = sub["Product Name"].map(normalize_name)
    sub = sub[sub["Normalized Name"] != ""]
    sub = sub[~sub["Customer"].str.upper().isin(AGGREGATE_CUSTOMERS)]
    sub["Promo Discount"] = pd.to_numeric(sub["Promo Discount"], errors="coerce")
    sub = sub.dropna(subset=["Promo Discount"])
    sub = sub[sub["Promo Discount"] > 0]
    if sub.empty:
        continue
    sub = sub.drop_duplicates(subset=["Normalized Name", "Customer", "Promo Discount"])
    sub["sheet"] = sheet
    sub["period_start"] = sheet_periods[sheet][0]
    sub["period_end"] = sheet_periods[sheet][1]
    promo_parts.append(sub)

promo_long = pd.concat(promo_parts, ignore_index=True)
print(f"{len(promo_long)} (normalized product, customer, promo, sheet) observations")
print("customers seen:", sorted(promo_long["Customer"].unique()))
promo_long.head()

3016 (normalized product, customer, promo, sheet) observations
customers seen: ['All', 'Heybox', 'Sonkwo']


,Product Name,Item Number,Promo Discount,Customer,Normalized Name,sheet,period_start,period_end
0,SPY×ANYA: Operation Memories,E05697,0.3000,All,spy_anya_operation_memories,11月,2024-11-01,2024-11-30
1,.hack//G.U. Last Recode,E05367,0.6835,All,hack_g_u_last_recode,11月,2024-11-01,2024-11-30
2,Accel World VS. Sword Art Online Deluxe Edition,E02452,0.6800,All,accel_world_vs_sword_art_online_deluxe,11月,2024-11-01,2024-11-30
3,Accel World VS. Sword Art Online Deluxe Edition,E02452,0.6835,All,accel_world_vs_sword_art_online_deluxe,11月,2024-11-01,2024-11-30
4,Ace Combat 7: Skies Unknown,E03174,0.6716,All,ace_combat_7_skies_unknown,11月,2024-11-01,2024-11-30


In [10]:
promo_long["_idx"] = promo_long["sheet"].map(sheet_idx)
promo_long = promo_long.sort_values(
    ["Normalized Name", "Customer", "Promo Discount", "_idx"]
).reset_index(drop=True)

promo_runs = []
group_keys = ["Normalized Name", "Customer", "Promo Discount"]
for (norm, customer, promo), grp in promo_long.groupby(group_keys, sort=False):
    grp = grp.sort_values("_idx")
    run_start = run_end = None
    prev_idx = None
    for _, row in grp.iterrows():
        if prev_idx is None:
            run_start = row["period_start"]
            run_end = row["period_end"]
            prev_idx = row["_idx"]
            continue
        if row["_idx"] == prev_idx + 1:
            run_end = row["period_end"]
        else:
            promo_runs.append((norm, customer, promo, run_start, run_end))
            run_start = row["period_start"]
            run_end = row["period_end"]
        prev_idx = row["_idx"]
    if prev_idx is not None:
        promo_runs.append((norm, customer, promo, run_start, run_end))

promo_history = pd.DataFrame(
    promo_runs,
    columns=["Normalized Name", "Customer", "Promo Discount", "start_date", "end_date"],
)
promo_history.insert(0, "Product Name", promo_history["Normalized Name"].map(display_names))
promo_history["start_month"] = promo_history["start_date"].dt.strftime("%Y-%m")
promo_history["end_month"] = promo_history["end_date"].dt.strftime("%Y-%m")
promo_history = promo_history[
    ["Product Name", "Normalized Name", "Customer", "Promo Discount",
     "start_month", "end_month", "start_date", "end_date"]
].sort_values(["Product Name", "Customer", "start_date", "Promo Discount"]).reset_index(drop=True)

print(f"{len(promo_history)} promo runs")
print(f"  across {promo_history['Normalized Name'].nunique()} products")
print(f"  across {promo_history[['Normalized Name','Customer']].drop_duplicates().shape[0]} (product, customer) pairs")

out = Path("product_promo_history.csv")
promo_history.drop(columns=["start_date", "end_date"]).to_csv(out, index=False)
print(f"Wrote {out.resolve()}")
promo_history.head(20)

1249 promo runs
  across 169 products
  across 449 (product, customer) pairs
Wrote /Users/christopherdrews/dev/bandai-data-analysis/product_promo_history.csv


,Product Name,Normalized Name,Customer,Promo Discount,start_month,end_month,start_date,end_date
0,.hack//G.U. Last Recode,hack_g_u_last_recode,All,0.6835,2024-11,2024-11,2024-11-01,2024-11-30
1,.hack//G.U. Last Recode,hack_g_u_last_recode,All,0.9000,2024-12,2025-03,2024-12-01,2025-03-31
2,.hack//G.U. Last Recode,hack_g_u_last_recode,All,0.8000,2025-04,2025-04,2025-04-01,2025-04-30
3,.hack//G.U. Last Recode,hack_g_u_last_recode,All,0.9000,2025-06,2025-06,2025-06-01,2025-06-30
4,.hack//G.U. Last Recode,hack_g_u_last_recode,Heybox,0.9000,2025-07,2025-09,2025-07-01,2025-09-30
5,.hack//G.U. Last Recode,hack_g_u_last_recode,Heybox,0.8000,2025-10,2026-04,2025-10-01,2026-04-30
6,.hack//G.U. Last Recode,hack_g_u_last_recode,Heybox,0.9000,2026-04,2026-04,2026-04-01,2026-04-30
7,.hack//G.U. Last Recode,hack_g_u_last_recode,Sonkwo,0.9000,2025-07,2025-07,2025-07-01,2025-07-31
8,.hack//G.U. Last Recode,hack_g_u_last_recode,Sonkwo,0.8000,2025-11,2026-02,2025-11-01,2026-02-28
9,ACE COMBAT 7: SKIES UNKNOWN,ace_combat_7_skies_unknown,All,0.6716,2024-11,2024-11,2024-11-01,2024-11-30


## Task 4 — Resolve Playasia PAX codes by name

`lookup_pax_codes.py` queries the Playasia `/products/filtered` endpoint (type=`digital codes`, digital=1) for each unique Product Name and writes `distinct_products_with_pax.csv` with the top match. This cell loads that output and surfaces rows that need attention: ones with no result, ones where the CSV's existing Item Number differs from the Playasia PAX code, and the original `0` Item Number rows.

In [11]:
pax = pd.read_csv('distinct_products_with_pax.csv', dtype=str).fillna('')
pax['lookup_score'] = pd.to_numeric(pax['lookup_score'], errors='coerce')
print('match_status counts:')
print(pax['match_status'].value_counts().to_string())
print(f"\nmean lookup_score: {pax['lookup_score'].mean():.1f}")
pax.head()

match_status counts:
match_status
differs        177
not_found       63
csv_invalid      2

mean lookup_score: 51.6


,Product Name,Item Number,lookup_pax_code,lookup_label,lookup_total_count,lookup_score,match_status
0,.hack//G.U. Last Recode,E05367,PAX0011620588,.Hack//G.U. Last Recode,1,95.7,differs
1,ACE COMBAT 7: SKIES UNKNOWN,E03174,PAX0012109542,Ace Combat 7: Skies Unknown,7,37.0,differs
2,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05445,PAX0012537704,Ace Combat 7: Skies Unknown - Top Gun: Maveric...,1,61.8,differs
3,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05444,,,0,0.0,not_found
4,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05445,PAX0012537704,Ace Combat 7: Skies Unknown - Top Gun: Maveric...,1,61.3,differs


### Rows with no Playasia match

In [12]:
pax[pax['match_status'] == 'not_found'][['Product Name', 'Item Number']]

,Product Name,Item Number
3,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05444
5,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05444
8,ARMORED CORE VI FIRES OF RUBICON Standard Edition,E05837
37,DEATH NOTE Killer Within,E06643
38,DEATH NOTE Killer Within Special Edition,E06646
...,...,...
235,That Time I Got Reincarnated as a Slime ISEKAI...,E06534
236,That Time I Got Reincarnated as a Slime ISEKAI...,E06534
238,Towa and the Guardians of the Sacred Tree Delu...,E06936
239,Towa and the Guardians of the Sacred Tree Delu...,L00117


### Rows where the CSV's Item Number was missing (`0`) — recovered PAX codes

In [13]:
pax[pax['match_status'] == 'csv_invalid'][['Product Name', 'Item Number', 'lookup_pax_code', 'lookup_label', 'lookup_score']]

,Product Name,Item Number,lookup_pax_code,lookup_label,lookup_score
72,Digimon Story Time Stranger Deluxe Edition,0,PAX0014928822,Digimon Story Time Stranger (Deluxe Edition),97.7
74,Digimon Story Time Stranger Ultimate Edition,0,PAX0014928856,Digimon Story Time Stranger (Ultimate Edition),97.8


### Lowest-confidence "differs" rows (sorted by fuzzy score — review these manually)

In [14]:
(pax[pax['match_status'] == 'differs']
 .sort_values('lookup_score')
 [['Product Name', 'Item Number', 'lookup_pax_code', 'lookup_label', 'lookup_score', 'lookup_total_count']]
 .head(30))

,Product Name,Item Number,lookup_pax_code,lookup_label,lookup_score,lookup_total_count
154,SCARLET NEXUS,E03557,PAX0011642246,Scarlet Nexus (Deluxe Edition),24.5,3
43,DORAEMON STORY OF SEASONS,E06163,PAX0012561776,Doraemon Story of Seasons: Friends of the Grea...,24.5,2
153,SCARLET NEXUS,E02985,PAX0011642246,Scarlet Nexus (Deluxe Edition),24.5,3
42,DORAEMON STORY OF SEASONS,E03496,PAX0012561776,Doraemon Story of Seasons: Friends of the Grea...,24.5,2
102,MOBILE SUIT GUNDAM SEED BATTLE DESTINY REMASTERED,E06942,PAX0014731554,Mobile Suit Gundam Seed Battle Destiny Remastered,26.5,1
36,DARK SOULS™: REMASTERED,E02716,PAX0010382852,Dark Souls: Remastered,26.7,5
39,DIGIMON STORY TIME STRANGER,E03053,PAX0014928856,Digimon Story Time Stranger (Ultimate Edition),26.8,5
31,DARK SOULS: REMASTERED,E02716,PAX0010382852,Dark Souls: Remastered,27.3,5
109,NARUTO TO BORUTO: SHINOBI STRIKER,E01572,PAX0009688844,Naruto to Boruto: Shinobi Striker,27.3,12
110,NARUTO TO BORUTO: SHINOBI STRIKER,E01927,PAX0009688844,Naruto to Boruto: Shinobi Striker,27.3,12


## Task 5 — Match Bandai PAX catalog by normalized name

Read `bandai_products.csv` (Bandai's PAX-code master list), add the same `Normalized Name` slug used elsewhere, and write it back in place. Then outer-merge with `distinct_products.csv` on `Normalized Name` to produce `distinct_products_bandai_match.csv` — one row per (royalty-report product × matching PAX entry), plus unmatched rows from either side.

In [15]:
bandai = pd.read_csv("bandai_products.csv", dtype=str)
bandai["Normalized Name"] = bandai["label"].map(normalize_name)
bandai.to_csv("bandai_products.csv", index=False)
print(f"bandai_products.csv: {len(bandai)} rows, {bandai['Normalized Name'].nunique()} distinct normalized names")
bandai.head()

bandai_products.csv: 665 rows, 313 distinct normalized names


,paxCode,label,Normalized Name
0,PAX0005145560,Dark Souls (Prepare to Die Edition),dark_souls_prepare_to_die
1,PAX0005558388,Naruto Shippuden: Ultimate Ninja Storm 3 Full ...,naruto_shippuden_ultimate_ninja_storm_3_full_b...
2,PAX0006085660,Naruto Shippuden: Ultimate Ninja Storm 3 Full ...,naruto_shippuden_ultimate_ninja_storm_3_full_b...
3,PAX0006208060,Dark Souls II,dark_souls_ii
4,PAX0006208094,Dark Souls II (Black Armour Edition),dark_souls_ii_black_armour


In [16]:
from difflib import SequenceMatcher

SIMILARITY_THRESHOLD = 0.95

# Qualifier tokens that mark an "Edition" SKU sitting on top of a base game.
# Order matters: longer multi-token qualifiers first so e.g.
# "tekken_8_season_2_deluxe" strips "_deluxe" before "_season_2".
QUALIFIER_SUFFIXES = (
    "season_2_deluxe", "season_2_ultimate",
    "season_3_deluxe", "season_3_ultimate",
    "season_2", "season_3",
    "deluxe", "ultimate", "special", "master", "premium",
    "legendary", "platinum", "daima", "advanced", "gold",
)

def find_base_game_match(slug: str, bandai_set: set[str]) -> tuple[str, list[str]] | None:
    """Iteratively strip qualifier suffixes from `slug` until it matches a
    Bandai slug. Returns (matched_base_slug, qualifiers_stripped) or None.
    """
    current, stripped = slug, []
    while current:
        if current in bandai_set:
            return current, stripped
        matched = False
        for qual in QUALIFIER_SUFFIXES:
            if current.endswith(f"_{qual}"):
                current = current[: -(len(qual) + 1)]
                stripped.append(qual)
                matched = True
                break
        if not matched:
            return None
    return None


merged = distinct.merge(
    bandai,
    on="Normalized Name",
    how="outer",
    indicator="match_status",
)
merged["match_status"] = merged["match_status"].map({
    "both": "matched",
    "left_only": "distinct_only",
    "right_only": "bandai_only",
})
merged["lookup_score"] = pd.NA
merged["lookup_bandai_slug"] = pd.NA

bandai_slugs = bandai["Normalized Name"].dropna().unique().tolist()
bandai_slug_set = set(bandai_slugs)
bandai_by_slug = bandai.groupby("Normalized Name")[["paxCode", "label"]]

# --- Pass 1: SequenceMatcher similarity for spelling-variant matches (≥0.95).
unmatched = (
    merged[merged["match_status"] == "distinct_only"]
    [["Normalized Name", "Product Name", "Item Number"]]
    .drop_duplicates(subset=["Normalized Name"])
)

similarity_rows = []
for _, row in unmatched.iterrows():
    slug = row["Normalized Name"]
    if not isinstance(slug, str) or not slug:
        continue
    best, best_score = None, 0.0
    for cand in bandai_slugs:
        score = SequenceMatcher(None, slug, cand).ratio()
        if score > best_score:
            best, best_score = cand, score
    if best is None or best_score < SIMILARITY_THRESHOLD:
        continue
    for _, bandai_row in bandai_by_slug.get_group(best).iterrows():
        similarity_rows.append({
            "Normalized Name": slug,
            "Product Name": row["Product Name"],
            "Item Number": row["Item Number"],
            "paxCode": bandai_row["paxCode"],
            "label": bandai_row["label"],
            "match_status": "similarity_match",
            "lookup_score": round(best_score, 3),
            "lookup_bandai_slug": best,
        })

# --- Pass 2: base-game fallback for Edition-suffix slugs. Only run on rows
# the similarity pass didn't already pick up.
promoted_by_similarity = {r["Normalized Name"] for r in similarity_rows}
base_game_rows = []
for _, row in unmatched.iterrows():
    slug = row["Normalized Name"]
    if slug in promoted_by_similarity:
        continue
    if not isinstance(slug, str) or not slug:
        continue
    result = find_base_game_match(slug, bandai_slug_set)
    if result is None:
        continue
    base_slug, stripped = result
    for _, bandai_row in bandai_by_slug.get_group(base_slug).iterrows():
        base_game_rows.append({
            "Normalized Name": slug,
            "Product Name": row["Product Name"],
            "Item Number": row["Item Number"],
            "paxCode": bandai_row["paxCode"],
            "label": bandai_row["label"],
            "match_status": "base_game_match",
            "lookup_score": pd.NA,
            "lookup_bandai_slug": base_slug,
        })

# Drop promoted distinct_only rows and append the new typed rows.
promoted_all = promoted_by_similarity | {r["Normalized Name"] for r in base_game_rows}
if promoted_all:
    keep_mask = ~(
        (merged["match_status"] == "distinct_only")
        & (merged["Normalized Name"].isin(promoted_all))
    )
    merged = pd.concat(
        [merged[keep_mask], pd.DataFrame(similarity_rows + base_game_rows)],
        ignore_index=True,
    )

merged = merged[
    ["Normalized Name", "Product Name", "Item Number", "paxCode", "label",
     "match_status", "lookup_score", "lookup_bandai_slug"]
].sort_values(["match_status", "Normalized Name"]).reset_index(drop=True)

print("merge breakdown:")
print(merged["match_status"].value_counts().to_string())

sim = merged[merged["match_status"] == "similarity_match"][
    ["Product Name", "Normalized Name", "lookup_bandai_slug", "lookup_score"]
].drop_duplicates(subset=["Normalized Name"])
print(f"\nsimilarity assignments (≥{SIMILARITY_THRESHOLD}): {len(sim)}")
print(sim.to_string(index=False) if len(sim) else "  (none)")

base = merged[merged["match_status"] == "base_game_match"][
    ["Product Name", "Normalized Name", "lookup_bandai_slug"]
].drop_duplicates(subset=["Normalized Name"])
print(f"\nbase-game fallbacks: {len(base)}")
print(base.to_string(index=False) if len(base) else "  (none)")

out = Path("distinct_products_bandai_match.csv")
merged.to_csv(out, index=False)
print(f"\nWrote {out.resolve()}")

# Filtered view: drop bandai_only rows so the file is just the royalty
# catalog with its exact / similarity / base_game / unmatched assignments.
royalty_only = merged[merged["match_status"] != "bandai_only"]
out_filtered = Path("royalty_pax_match.csv")
royalty_only.to_csv(out_filtered, index=False)
print(f"Wrote {out_filtered.resolve()} ({len(royalty_only)} rows, bandai_only excluded)")

# Per-slug match method for the distinct catalog. Priority: exact > similarity > base_game > unmatched.
def _classify(slug: str) -> str:
    statuses = set(merged.loc[merged["Normalized Name"] == slug, "match_status"])
    if "matched" in statuses:
        return "exact"
    if "similarity_match" in statuses:
        return "similarity"
    if "base_game_match" in statuses:
        return "base_game"
    return "unmatched"

# paxCode lookup: exact + similarity + base_game all flow into the joined list.
pax_per_slug = (
    merged[merged["match_status"].isin(["matched", "similarity_match", "base_game_match"])]
    .groupby("Normalized Name")["paxCode"]
    .agg(lambda s: ", ".join(sorted(set(s.dropna()))))
)

# 1. distinct_products.csv — paxCode + match_method per product.
distinct_with_pax = distinct.copy()
distinct_with_pax["paxCode"] = (
    distinct_with_pax["Normalized Name"].map(pax_per_slug).fillna("")
)
distinct_with_pax["match_method"] = distinct_with_pax["Normalized Name"].map(_classify)
distinct_with_pax.to_csv("distinct_products.csv", index=False)
counts = distinct_with_pax["match_method"].value_counts().to_dict()
print(
    f"\nUpdated distinct_products.csv "
    f"(exact={counts.get('exact',0)}, "
    f"similarity={counts.get('similarity',0)}, "
    f"base_game={counts.get('base_game',0)}, "
    f"unmatched={counts.get('unmatched',0)})"
)

# 2. product_srp_history.csv — paxCode on each run row.
srp_with_pax = srp_history.drop(columns=["start_date", "end_date"]).copy()
srp_with_pax["paxCode"] = srp_with_pax["Normalized Name"].map(pax_per_slug).fillna("")
srp_with_pax.to_csv("product_srp_history.csv", index=False)
print(f"Updated product_srp_history.csv "
      f"({(srp_with_pax['paxCode'] != '').sum()} / {len(srp_with_pax)} runs matched)")

# 3. product_promo_history.csv — same enrichment.
promo_with_pax = promo_history.drop(columns=["start_date", "end_date"]).copy()
promo_with_pax["paxCode"] = promo_with_pax["Normalized Name"].map(pax_per_slug).fillna("")
promo_with_pax.to_csv("product_promo_history.csv", index=False)
print(f"Updated product_promo_history.csv "
      f"({(promo_with_pax['paxCode'] != '').sum()} / {len(promo_with_pax)} runs matched)")

merged.head(15)

merge breakdown:
match_status
bandai_only         354
matched             311
base_game_match      66
distinct_only        14
similarity_match      7

similarity assignments (≥0.95): 4
                    Product Name                  Normalized Name                lookup_bandai_slug lookup_score
      Everybody's Golf: Hotshots        everybody_s_golf_hotshots        everybody_s_golf_hot_shots         0.98
                    God Eaters 3                     god_eaters_3                       god_eater_3        0.957
SD GUNDAM G GENERATION CROSS RAY sd_gundam_g_generation_cross_ray sd_gundam_g_generation_cross_rays        0.985
    SPY×ANYA: Operation Memories      spy_anya_operation_memories     spy_x_anya_operation_memories        0.964

base-game fallbacks: 29
                                                                    Product Name                                                  Normalized Name                                        lookup_bandai_slug
               ACE CO

,Normalized Name,Product Name,Item Number,paxCode,label,match_status,lookup_score,lookup_bandai_slug
0,11_11_memories_retold,NaN,NaN,PAX0009838002,11-11 Memories Retold,bandai_only,<NA>,<NA>
1,11_11_memories_retold,NaN,NaN,PAX0011615896,11-11 Memories Retold,bandai_only,<NA>,<NA>
2,11_11_memories_retold,NaN,NaN,PAX0014311212,11-11 Memories Retold,bandai_only,<NA>,<NA>
3,ace_combat_7_skies_unknown_deluxe,NaN,NaN,PAX0009943572,Ace Combat 7: Skies Unknown (Deluxe Edition),bandai_only,<NA>,<NA>
4,ace_combat_7_skies_unknown_season_pass,NaN,NaN,PAX0009965366,Ace Combat 7: Skies Unknown Season Pass (DLC),bandai_only,<NA>,<NA>
5,ace_combat_7_skies_unknown_season_pass,NaN,NaN,PAX0010432186,Ace Combat 7: Skies Unknown Season Pass (DLC),bandai_only,<NA>,<NA>
6,ace_combat_7_skies_unknown_top_gun_maverick_ai...,NaN,NaN,PAX0013456622,Ace Combat 7: Skies Unknown - Top Gun: Maveric...,bandai_only,<NA>,<NA>
7,ace_combat_8_wings_of_theve,NaN,NaN,PAX0015391392,Ace Combat 8: Wings Of Theve,bandai_only,<NA>,<NA>
8,ace_combat_assault_horizon_enhanced,NaN,NaN,PAX0006212072,Ace Combat Assault Horizon (Enhanced Edition),bandai_only,<NA>,<NA>
9,arcade_game_series_3_in_1_pack,NaN,NaN,PAX0011537730,Arcade Game Series: 3 in 1 Pack,bandai_only,<NA>,<NA>
